# Multilayer Perceptron Evaluation

This notebook tests whether a small neural network captures nonlinear structure in EPL remaining-points forecasts. Experiments A, B, and C use the same predictor progression, validation folds, test seasons, and performance metrics as the primary analysis.

Predictors are standardized inside the pipeline for every training fold. Network architecture and L2 regularization strength are selected using validation MSE, and convergence warnings are counted as a model-fitting check.


## 1. Imports and configuration


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV

RANDOM_STATE = 4031
DEVELOPMENT_SEASONS = list(range(2016, 2023))
TEST_SEASONS = list(range(2023, 2026))

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")



import warnings

from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


## 2. Load data and define experiments


In [2]:
def locate_dataset(filename="epl_merged_data.xlsx"):
    for root in [Path.cwd(), *Path.cwd().parents]:
        candidate = root / "outputs" / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate outputs/{filename} from {Path.cwd().resolve()}"
    )


dataset_path = locate_dataset()
epl_data = pd.read_excel(dataset_path)

required_columns = {
    "Season", "Team", "Promoted", "Curr_Points", "EOS_Total_Points"
}
missing_required = sorted(required_columns.difference(epl_data.columns))
if missing_required:
    raise KeyError(f"Missing required columns: {missing_required}")
if epl_data.duplicated(["Season", "Team"]).any():
    raise ValueError("The dataset contains duplicate Season-Team observations.")

epl_data["Remaining_Points"] = (
    epl_data["EOS_Total_Points"] - epl_data["Curr_Points"]
)

CURRENT_PERFORMANCE_FEATURES = [
    "Curr_Avg_Win_Percentage",
    "Curr_Avg_Draw_Percentage",
    "Curr_Avg_Goals",
    "Curr_Avg_SOT",
    "Curr_Avg_Opp_Goals",
    "Curr_Avg_Opp_SOT",
    "Curr_Avg_xG",
    "Curr_Avg_xGA",
    "Curr_Avg_Deep",
    "Curr_Avg_Deep_Allowed",
    "Curr_Points",
]
PREVIOUS_SEASON_FEATURES = [
    "Prev_Total_Points",
    "Prev_Avg_Goals",
    "Prev_Avg_SOT",
    "Prev_Avg_Opp_Goals",
    "Prev_Avg_Opp_SOT",
    "Prev_Avg_xG",
    "Prev_Avg_xGA",
    "Prev_Points",
]
TRANSFER_FEATURES = [
    "Fee_Spending",
    "Fee_Earnings",
    "Permanent_In_Count",
    "Permanent_Out_Count",
]

EXPERIMENTS = {
    "Experiment A: Current performance": CURRENT_PERFORMANCE_FEATURES,
    "Experiment B: Current + previous season": (
        CURRENT_PERFORMANCE_FEATURES + PREVIOUS_SEASON_FEATURES + ["Promoted"]
    ),
    "Experiment C: Current + previous + transfers": (
        CURRENT_PERFORMANCE_FEATURES
        + PREVIOUS_SEASON_FEATURES
        + ["Promoted"]
        + TRANSFER_FEATURES
    ),
}
TARGET = "Remaining_Points"

for experiment_name, features in EXPERIMENTS.items():
    missing = sorted(set(features).difference(epl_data.columns))
    if missing:
        raise KeyError(f"{experiment_name} is missing predictors: {missing}")
    if epl_data[features].isna().any().any():
        raise ValueError(f"{experiment_name} contains missing predictor values.")

print(f"Loaded: {dataset_path.resolve()}")
print(f"Observations: {len(epl_data):,}")
for experiment_name, features in EXPERIMENTS.items():
    print(f"{experiment_name}: {len(features)} predictors")


Loaded: C:\Users\micha\epl_results_prediction\outputs\epl_merged_data.xlsx
Observations: 200
Experiment A: Current performance: 11 predictors
Experiment B: Current + previous season: 20 predictors
Experiment C: Current + previous + transfers: 24 predictors


## 3. Create development, validation, and test samples


In [3]:
development = (
    epl_data.loc[epl_data["Season"].isin(DEVELOPMENT_SEASONS)]
    .sort_values(["Season", "Team"])
    .reset_index(drop=True)
)
test_data = (
    epl_data.loc[epl_data["Season"].isin(TEST_SEASONS)]
    .sort_values(["Season", "Team"])
    .reset_index(drop=True)
)

if development.empty or test_data.empty:
    raise ValueError("Development or test data is empty.")

# Grouped leave-one-season-out validation keeps every season intact.
cv_splits = []
for validation_season in DEVELOPMENT_SEASONS:
    train_indices = np.flatnonzero(development["Season"].ne(validation_season))
    validation_indices = np.flatnonzero(
        development["Season"].eq(validation_season)
    )
    if not len(validation_indices):
        raise ValueError(f"No validation observations for season {validation_season}.")
    cv_splits.append((train_indices, validation_indices))

print(f"Development observations: {len(development):,}")
print(f"Validation folds: {len(cv_splits)}")
print(f"Test observations: {len(test_data):,}")


Development observations: 140
Validation folds: 7
Test observations: 60


## 4. Configure the MLP pipeline


In [4]:
mlp_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "model",
            MLPRegressor(
                activation="relu",
                solver="lbfgs",
                max_iter=1_500,
                max_fun=5_000,
                tol=1e-3,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
parameter_grid = {
    "model__hidden_layer_sizes": [(4,), (8,), (16,), (8, 4)],
    "model__alpha": [0.1, 1.0, 10.0],
}
SCORING = {
    "mse": "neg_mean_squared_error",
    "mae": "neg_mean_absolute_error",
}

combination_count = int(
    np.prod([len(values) for values in parameter_grid.values()])
)
print(f"MLP hyperparameter combinations: {combination_count}")


MLP hyperparameter combinations: 12


## 5. Define the evaluation procedure


In [5]:
def evaluate_mlp(experiment_name, features):
    X_development = development[features]
    y_development = development[TARGET]
    convergence_warning_count = 0

    search = GridSearchCV(
        estimator=clone(mlp_pipeline),
        param_grid=parameter_grid,
        scoring=SCORING,
        refit="mse",
        cv=cv_splits,
        n_jobs=1,
        error_score="raise",
    )
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always", ConvergenceWarning)
        search.fit(X_development, y_development)
    convergence_warning_count += sum(
        issubclass(item.category, ConvergenceWarning) for item in captured
    )

    best_grid_row = pd.DataFrame(search.cv_results_).iloc[search.best_index_]
    validation_mse = -best_grid_row["mean_test_mse"]
    validation_mae = -best_grid_row["mean_test_mae"]
    test_actual = test_data[TARGET]
    test_predicted = search.best_estimator_.predict(test_data[features])
    test_mse = mean_squared_error(test_actual, test_predicted)

    return {
        "Experiment": experiment_name,
        "Predictors": len(features),
        "Best_Parameters": search.best_params_,
        "Validation_MSE": validation_mse,
        "Validation_MAE": validation_mae,
        "Validation_RMSE": np.sqrt(validation_mse),
        "Test_MSE": test_mse,
        "Test_MAE": mean_absolute_error(test_actual, test_predicted),
        "Test_RMSE": np.sqrt(test_mse),
        "Convergence_Warnings": convergence_warning_count,
    }


## 6. Fit experiments and report performance


In [6]:
mlp_results = []
for experiment_name, features in EXPERIMENTS.items():
    print(f"Fitting {experiment_name} ...")
    mlp_results.append(evaluate_mlp(experiment_name, features))

performance_summary = pd.DataFrame(
    [
        {
            "Experiment": result["Experiment"],
            "Predictors": result["Predictors"],
            "Validation_MSE": result["Validation_MSE"],
            "Validation_MAE": result["Validation_MAE"],
            "Validation_RMSE": result["Validation_RMSE"],
            "Test_MSE": result["Test_MSE"],
            "Test_MAE": result["Test_MAE"],
            "Test_RMSE": result["Test_RMSE"],
            "Architecture": str(
                result["Best_Parameters"]["model__hidden_layer_sizes"]
            ),
            "Alpha": result["Best_Parameters"]["model__alpha"],
            "Convergence_Warnings": result["Convergence_Warnings"],
        }
        for result in mlp_results
    ]
).sort_values("Validation_MSE").reset_index(drop=True)

display(performance_summary)


Fitting Experiment A: Current performance ...


Fitting Experiment B: Current + previous season ...


Fitting Experiment C: Current + previous + transfers ...


,Experiment,Predictors,Validation_MSE,Validation_MAE,Validation_RMSE,Test_MSE,Test_MAE,Test_RMSE,Architecture,Alpha,Convergence_Warnings
0,Experiment A: Current performance,11,105.727,8.482,10.282,132.695,9.215,11.519,"(4,)",10.000,47
1,Experiment B: Current + previous season,20,132.411,9.005,11.507,171.069,10.430,13.079,"(16,)",10.000,63
2,Experiment C: Current + previous + transfers,24,138.248,9.304,11.758,201.971,11.664,14.212,"(8,)",10.000,62
